# run_baseline_comparison_gate

该 Notebook 是 `baseline_comparison_gate` 的 Colab 执行入口。它只负责 Colab 会话编排, 正式逻辑全部委托给仓库脚本和 `experiments/baseline_comparison_gate/` 模块。

当前 Notebook 的目标是完成冷启动 smoke 验证: 拉取外部 baseline 上游源码、执行 source probe、执行 preflight、运行 adapter skeleton smoke, 并在成功后将完整结果包复制到 Google Drive。当前 smoke 结果不能支持论文 claim。

## 0. 使用边界

- 本 Notebook 不直接写出正式 `records`、`thresholds`、`tables`、`figures` 或 `reports` 逻辑。
- 本 Notebook 不把 `external_baselines/` 提交进仓库。
- `external_baselines/` 是冷启动缓存目录, 每次 Colab 运行时通过 `scripts/prepare_baselines/fetch_external_baselines.py` 重新拉取。
- 真实 baseline adapter、权重 digest 和正式比较表仍需要后续阶段继续实现。

In [ ]:
# 可修改配置区
REPO_URL = "https://github.com/RICHAAARC/TSTW-VW.git"  # 示例: "https://github.com/<user>/<repo>.git"。若当前 Colab 已经在仓库中, 可留空。
REPO_DIR = "/content/TSTW-VW"
GIT_REF = "explicit-sync"  # 可改为具体分支、tag 或 commit。留空则不切换。
DRIVE_MOUNT = "/content/drive"
DRIVE_RESULT_ROOT = "/content/drive/MyDrive/TSTW/results"
SESSION_RUN_ROOT = "/content/TSTW_runtime/runs/baseline_comparison_smoke"
RUN_VIDEOSEAL_REAL_SMOKE = True
VIDEOSEAL_INPUT_VIDEO_PATH = ""  # 留空时脚本会生成一个确定性 smoke 输入视频。


In [ ]:
from pathlib import Path
import os
import subprocess
import sys


def run_command(command, cwd=None):
    """运行命令并在失败时立即停止 Notebook。"""
    print("$", " ".join(command))
    completed = subprocess.run(command, cwd=cwd, text=True)
    if completed.returncode != 0:
        raise RuntimeError(f"命令失败: {' '.join(command)}")


def run_python_script(script, *args):
    """使用当前 Python 解释器运行仓库脚本。"""
    run_command([sys.executable, script, *args], cwd=REPO_DIR)


## 1. 挂载 Google Drive

该步骤只负责挂载 Drive。结果目录不会在这里提前创建, 只有 smoke 成功后才会复制完整结果包。

In [ ]:
try:
    from google.colab import drive
    drive.mount(DRIVE_MOUNT)
except Exception as exc:
    print(f"非 Colab 环境或 Drive 挂载失败: {exc}")


## 2. 获取项目仓库

如果 `REPO_URL` 为空, Notebook 假设 `/content/TSTW-VW` 已经存在。否则会从远程仓库冷启动 clone。

In [ ]:
repo_path = Path(REPO_DIR)
if REPO_URL and not repo_path.exists():
    run_command(["git", "clone", REPO_URL, REPO_DIR])
elif not repo_path.exists():
    raise FileNotFoundError("REPO_DIR 不存在, 请设置 REPO_URL 或先上传/克隆仓库。")

os.chdir(REPO_DIR)
if GIT_REF:
    run_command(["git", "fetch", "--all", "--tags"], cwd=REPO_DIR)
    run_command(["git", "checkout", GIT_REF], cwd=REPO_DIR)

print(f"当前仓库目录: {Path.cwd()}")
run_command(["git", "branch", "--show-current"], cwd=REPO_DIR)
run_command(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR)


## 3. 拉取外部 baseline 源码

`external_baselines/` 被 `.gitignore` 排除是合理的: 它属于第三方上游源码缓存, 不能作为本项目提交内容。每次 Colab 冷启动都通过下面的脚本按 manifest 中固定的 commit 重新拉取。

In [ ]:
run_python_script("scripts/prepare_baselines/fetch_external_baselines.py", "--print-plan")


## 3.1 安装 VideoSeal smoke 依赖

该步骤只安装 `external_videoseal` 真实 smoke 所需依赖。`external_baselines/` 仍然只是运行时缓存目录, 不会进入 git 提交。


In [ ]:
if RUN_VIDEOSEAL_REAL_SMOKE:
    run_command([
        sys.executable, "-m", "pip", "install",
        "-r", "external_baselines/external_videoseal/upstream/requirements.txt"
    ], cwd=REPO_DIR)


## 4. Source probe

该步骤检查上游源码树是否包含预期入口线索。它不下载权重, 也不运行真实模型。

In [ ]:
run_python_script("scripts/prepare_baselines/probe_external_baseline_sources.py")


## 5. Colab preflight

该步骤会显示当前真实 baseline 仍然缺少哪些条件, 包括权重 digest 和真实 adapter smoke。

In [ ]:
run_python_script("scripts/prepare_baselines/check_baseline_colab_preflight.py")


## 6. 运行 baseline smoke 并成功后落盘到 Drive

该命令先写入 `/content/TSTW_runtime/runs/`, 只有 smoke 成功且本地结果完整后, 才复制到 `/content/drive/MyDrive/TSTW/results/baseline_comparison_gate/<RUN_ID>/`。

In [ ]:
run_python_script(
    "scripts/prepare_baselines/run_baseline_comparison_smoke.py",
    "--run-root", SESSION_RUN_ROOT,
    "--result-root", DRIVE_RESULT_ROOT,
)


## 8. external_videoseal 真实可运行性 smoke

该步骤会真实加载 VideoSeal、下载权重、计算权重 digest、执行单视频 embed / detect, 并额外执行 H.264 CRF 28 后的 detect。该结果仍然只是单 baseline smoke, 不能替代正式 fixed-FPR baseline comparison。


In [ ]:
if RUN_VIDEOSEAL_REAL_SMOKE:
    videoseal_args = [
        "--run-root", "/content/TSTW_runtime/runs/external_videoseal_real_smoke",
        "--result-root", DRIVE_RESULT_ROOT,
    ]
    if VIDEOSEAL_INPUT_VIDEO_PATH:
        videoseal_args.extend(["--input-video-path", VIDEOSEAL_INPUT_VIDEO_PATH])
    run_python_script("scripts/prepare_baselines/run_videoseal_real_smoke.py", *videoseal_args)
else:
    print("已跳过 external_videoseal 真实 smoke。")


## 7. 当前结论

如果上面的所有单元执行成功, 说明阶段三的 Colab 冷启动工程链路可用。该结果仍然不能用于论文 claim。下一步应实现 `external_videoseal` 的真实 adapter、权重下载和权重 digest, 然后逐步扩展到 `external_rivagan` 与 `external_hidden_framewise`。